In [18]:
from langgraph.graph import StateGraph,START,END
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage,HumanMessage,BaseMessage
from pydantic import BaseModel,Field
from dotenv import load_dotenv
from typing import Literal,TypedDict,Annotated
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [19]:
load_dotenv()

True

In [20]:
llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=1.8)

In [21]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [22]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [23]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [24]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [25]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [26]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crumby and had a saucy attitude!',
 'explanation': 'Let\'s break down this joke to understand why it\'s amusing.\n\nThe joke is a play on words, using puns to create humor. It\'s asking why the pizza is in a bad mood and provides two interconnected reasons. \n\n1. "Feeling a little crumby": The term \'crumby\' usually means being of poor quality or feeling unwell. However, in this context, it\'s funny because a pizza can become crumby, as in, it can have crunchy bits or crumbs, especially if it\'s been broken apart. So it\'s saying the pizza feels unwell (crumby) and also referencing the fact that pizza, by nature, can produce crumbs.\n\n2. "Had a saucy attitude": Here, the jokester uses another dual-edged word, "saucy". In food, sauce is an essential component that\'s typically smooth, tangy or rich in flavor and found on top of the pizza. But when said about a person, "saucy" means they 

In [27]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crumby and had a saucy attitude!', 'explanation': 'Let\'s break down this joke to understand why it\'s amusing.\n\nThe joke is a play on words, using puns to create humor. It\'s asking why the pizza is in a bad mood and provides two interconnected reasons. \n\n1. "Feeling a little crumby": The term \'crumby\' usually means being of poor quality or feeling unwell. However, in this context, it\'s funny because a pizza can become crumby, as in, it can have crunchy bits or crumbs, especially if it\'s been broken apart. So it\'s saying the pizza feels unwell (crumby) and also referencing the fact that pizza, by nature, can produce crumbs.\n\n2. "Had a saucy attitude": Here, the jokester uses another dual-edged word, "saucy". In food, sauce is an essential component that\'s typically smooth, tangy or rich in flavor and found on top of the pizza. But when said about a person, 

In [28]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crumby and had a saucy attitude!', 'explanation': 'Let\'s break down this joke to understand why it\'s amusing.\n\nThe joke is a play on words, using puns to create humor. It\'s asking why the pizza is in a bad mood and provides two interconnected reasons. \n\n1. "Feeling a little crumby": The term \'crumby\' usually means being of poor quality or feeling unwell. However, in this context, it\'s funny because a pizza can become crumby, as in, it can have crunchy bits or crumbs, especially if it\'s been broken apart. So it\'s saying the pizza feels unwell (crumby) and also referencing the fact that pizza, by nature, can produce crumbs.\n\n2. "Had a saucy attitude": Here, the jokester uses another dual-edged word, "saucy". In food, sauce is an essential component that\'s typically smooth, tangy or rich in flavor and found on top of the pizza. But when said about a person,

In [29]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a complicated relationship! (get it?)',
 'explanation': 'A clever play on words. This joke is a great example of a humorous pun. Here\'s how it works:\n\nThe joke starts by setting up a unexpected scenario: spaghetti, an inanimate object and a type of food, is considering getting married. This already raises eyebrows and creates curiosity.\n\nThe punchline, "it was afraid of getting tangled up in a complicated relationship," is where the magic happens. The word "tangled" has a double meaning here:\n\n1. **Literal meaning**: Spaghetti is a long, thin, and flexible food that can easily become twisted and knotted, or "tangled" up.\n2. **Figurative meaning**: In relationships, "tangled up" is a common idiom for getting entangled in a complicated or messy situation, often emotional or confusing.\n\nBy using the word "tangled," the joke creates a clever connection betwe

In [30]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crumby and had a saucy attitude!', 'explanation': 'Let\'s break down this joke to understand why it\'s amusing.\n\nThe joke is a play on words, using puns to create humor. It\'s asking why the pizza is in a bad mood and provides two interconnected reasons. \n\n1. "Feeling a little crumby": The term \'crumby\' usually means being of poor quality or feeling unwell. However, in this context, it\'s funny because a pizza can become crumby, as in, it can have crunchy bits or crumbs, especially if it\'s been broken apart. So it\'s saying the pizza feels unwell (crumby) and also referencing the fact that pizza, by nature, can produce crumbs.\n\n2. "Had a saucy attitude": Here, the jokester uses another dual-edged word, "saucy". In food, sauce is an essential component that\'s typically smooth, tangy or rich in flavor and found on top of the pizza. But when said about a person, 

In [31]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crumby and had a saucy attitude!', 'explanation': 'Let\'s break down this joke to understand why it\'s amusing.\n\nThe joke is a play on words, using puns to create humor. It\'s asking why the pizza is in a bad mood and provides two interconnected reasons. \n\n1. "Feeling a little crumby": The term \'crumby\' usually means being of poor quality or feeling unwell. However, in this context, it\'s funny because a pizza can become crumby, as in, it can have crunchy bits or crumbs, especially if it\'s been broken apart. So it\'s saying the pizza feels unwell (crumby) and also referencing the fact that pizza, by nature, can produce crumbs.\n\n2. "Had a saucy attitude": Here, the jokester uses another dual-edged word, "saucy". In food, sauce is an essential component that\'s typically smooth, tangy or rich in flavor and found on top of the pizza. But when said about a person,